# E3.5 · The metrics that matter at your level

**Function E — Governance, Risk, Compliance & the CISO Office → The BISO, Risk Communicator & CISO Office**  ·  *Security of AI*

---

**Risk.** Reporting activity instead of exposure.

**Control.** Inventory coverage, attested-identity share, standing-access reduction, MTT-revoke, blast-radius distribution, eval-gate pass rate.

**This lab.** Instrument the six board-level metrics from your own lab stack.

| | |
|---|---|
| Open-source tooling | OpenSearch |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E3.5"))

The metrics that matter at CISO level are few, and none of them is a count of alerts.

In [ ]:
from cybercommons import grc, planes, redteam, sandbox
W = planes.Tool

# 1. exposure — how much unreviewed action exists
fleet = [planes.Manifest("a1", [W("read_file"), W("write_file", writes=True,
                                                  scope="project")], rung="L2.5"),
         planes.Manifest("a2", [W("deploy_prod", writes=True, scope="org",
                                  reversible=False)], rung="L2.5")]
exposure = sum(m.blast_radius()["total"] for m in fleet)

# 2. likelihood — measured, not asserted
def t(a):
    box = sandbox.default_sandbox()
    if a.surface != redteam.CONTAINMENT:
        return False, "n/a"
    tool = ("http_get" if a.payload.startswith("http")
            else "read_file" if a.payload.startswith("/") else a.payload)
    d = box.call(tool, a.payload if tool != a.payload else "")
    return d.allowed, d.reason
asr = redteam.run_campaign(t, "fleet").asr(redteam.CONTAINMENT)

# 3. assurance — how much is currently evidenced
import time
now = time.time()
tests = [grc.ControlTest(c, True, "auto", tested_at=now - 5 * 86400)
         for c in ("AC-1", "AC-2", "SB-1", "EV-1")]
cov = grc.verify_continuously(tests, [c.cid for c in grc.CATALOGUE], now=now)["coverage"]

print(f"exposure   fleet blast radius        {exposure}")
print(f"likelihood red-team ASR (containment) {asr:.0%}")
print(f"assurance  controls evidenced        {cov:.0%}")
print(f"coverage   agents in inventory       (report honestly — usually < 100%)")
print(f"speed      measured time-to-stop     seconds, from a game day")

Five numbers. Each moves when someone does work, and each degrades on its own if nobody does — which is the property that makes a metric worth reporting monthly.

### Expect

Exposure prints as a summed blast radius, containment ASR as 0%, and control coverage as 50%.

### Your turn

Which of the five can you produce today without a project? Start reporting that one monthly and let the missing ones become conspicuous.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E3.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*